In [ ]:
import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from analytical_cases import *

# Interactive widgets
import ipywidgets as widgets
from ipywidgets import interact, Layout, interactive
from IPython.display import display

# Avoid warnings for divisions by zero (e.g., r=0)
np.seterr(divide='ignore', invalid='ignore')

In [ ]:
def interactive_strain_plot(filename="output/fields.vtu",
                            analytical_case="lame",
                            tol=1e-5):
    """
    Interactive radial strain plot with dropdown menus for field and Z-level selection.
    Automatically detects all available strain fields in both point and cell data.
    """
    print(f"Loading data from: {filename}")
    grid = pv.read(filename)

    # --- Find all available strain fields from both point and cell data ---
    point_fields = {k: 'point' for k in grid.point_data if "Strain" in k}
    cell_fields = {k: 'cell' for k in grid.cell_data if "Strain" in k}
    all_strain_fields = {**point_fields, **cell_fields}

    if not all_strain_fields:
        print("[ERROR] No 'Strain' field found in the VTU file.")
        return
    
    print(f"Found {len(all_strain_fields)} strain fields.")

    # This is the outer function that gets called when you select a field
    def on_field_selected(strain_field_name):
        strain_field_source = all_strain_fields[strain_field_name]
        print(f"\nAnalyzing '{strain_field_name}' (from {strain_field_source}_data)...")

        # --- Select coordinate source based on the selected field ---
        if strain_field_source == "point":
            coords = grid.points
            e_tensor_full = grid.point_data[strain_field_name].reshape((-1, 3, 3))
        else:
            coords = grid.cell_centers().points
            e_tensor_full = grid.cell_data[strain_field_name].reshape((-1, 3, 3))

        unique_z_coords = np.unique(np.round(coords[:, 2], 6))
        print(f"Found {len(unique_z_coords)} unique Z levels.")

        # This is the inner function that gets called when you select a Z-level
        def plot_strain_at_z(z_level):
            mask = np.abs(coords[:, 2] - z_level) < tol
            if not np.any(mask):
                print(f"[WARNING] No elements found at z = {z_level} ± {tol}")
                return

            e_tensor = e_tensor_full[mask]
            x, y = coords[mask, 0], coords[mask, 1]
            r = np.sqrt(x**2 + y**2)
            theta = np.arctan2(y, x)

            c, s = np.cos(theta), np.sin(theta)
            c2, s2, sc = c**2, s**2, s*c

            e_xx, e_yy, e_xy, e_zz = e_tensor[:,0,0], e_tensor[:,1,1], e_tensor[:,0,1], e_tensor[:,2,2]
            eps_rr = e_xx * c2 + e_yy * s2 + 2 * e_xy * sc
            eps_tt = e_xx * s2 + e_yy * c2 - 2 * e_xy * sc
            eps_zz = e_zz

            df = pd.DataFrame({"r": np.round(r, 6), "eps_rr": eps_rr, "eps_tt": eps_tt, "eps_zz": eps_zz})
            grouped = df.groupby("r").mean().reset_index()
            r_avg = grouped["r"].values

            # Analytical solution
            if analytical_case is None:
                eps_tt_an, eps_rr_an = np.zeros_like(r_avg), np.zeros_like(r_avg)
            else:
                # Assuming analytical_cases.py is in the path
                u_r_an = radial_displacement(r_avg)
                eps_tt_an = u_r_an / r_avg
                eps_rr_an = radial_strain(r_avg)

            # Plot
            fig, ax = plt.subplots(figsize=(10, 7))
            ax.plot(r_avg, grouped["eps_rr"], 'o', c='darkgreen', label=f"ε_rr (FEM, {strain_field_source})")
            ax.plot(r_avg, grouped["eps_tt"], 's', c='crimson', label=f"ε_tt (FEM, {strain_field_source})")
            ax.plot(r_avg, grouped["eps_zz"], 'x', c='lightblue', label=f"ε_zz (FEM, {strain_field_source})")
            ax.plot(r_avg, eps_rr_an, '--', c='darkgreen', lw=2, label="ε_rr (Analytical)")
            ax.plot(r_avg, eps_tt_an, '--', c='crimson', lw=2, label="ε_tt (Analytical)")

            ax.set_xlabel("Radius r (m)", fontsize=12)
            ax.set_ylabel("Strain ε (-)", fontsize=12)
            ax.set_title(f"'{strain_field_name}' vs. radius at z = {z_level}", fontsize=14)
            ax.grid(True, linestyle=':')
            ax.legend()
            plt.tight_layout()
            plt.show()

        # Create the Z-level dropdown for the currently selected field
        interact(
            plot_strain_at_z,
            z_level=widgets.Dropdown(
                options=unique_z_coords,
                value=unique_z_coords[len(unique_z_coords) // 2],
                description='Select Z-level:',
                style={'description_width': 'initial'},
                layout=Layout(width='50%')
            )
        )

    # --- Create the TOP-LEVEL dropdown for selecting the strain field ---
    interact(
        on_field_selected,
        strain_field_name=widgets.Dropdown(
            options=list(all_strain_fields.keys()),
            description='Select Strain Field:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

interactive_strain_plot(
    filename="output/fields.vtu",
    analytical_case=None
)

In [ ]:
def interactive_stress_plot(filename="output/fields.vtu",
                            analytical_case="lame",
                            tol=1e-5):
    """
    Interactive radial stress plot with dropdown menus for field and Z-level selection.
    Automatically detects all available stress fields in both point and cell data.
    """
    print(f"Loading data from: {filename}")
    grid = pv.read(filename)

    # --- Find all available stress fields from both point and cell data ---
    point_fields = {k: 'point' for k in grid.point_data if "Stress" in k}
    cell_fields = {k: 'cell' for k in grid.cell_data if "Stress" in k}
    all_stress_fields = {**point_fields, **cell_fields}

    if not all_stress_fields:
        print("[ERROR] No 'Stress' field found in the VTU file.")
        return

    print(f"Found {len(all_stress_fields)} stress fields.")

    # --- Top-level dropdown: select stress field ---
    def on_field_selected(stress_field_name):
        stress_field_source = all_stress_fields[stress_field_name]
        print(f"\nAnalyzing '{stress_field_name}' (from {stress_field_source}_data)...")

        # Select coordinates
        if stress_field_source == "point":
            coords = grid.points
            s_tensor_full = grid.point_data[stress_field_name].reshape((-1, 3, 3))
        else:
            coords = grid.cell_centers().points
            s_tensor_full = grid.cell_data[stress_field_name].reshape((-1, 3, 3))

        unique_z_coords = np.unique(np.round(coords[:, 2], 6))
        print(f"Found {len(unique_z_coords)} unique Z levels.")

        # --- Inner plot function for selected z-level ---
        def plot_stress_at_z(z_level):
            mask = np.abs(coords[:, 2] - z_level) < tol
            if not np.any(mask):
                print(f"[WARNING] No elements found at z = {z_level} ± {tol}")
                return

            s_tensor = s_tensor_full[mask]
            x, y = coords[mask, 0], coords[mask, 1]
            r = np.sqrt(x**2 + y**2)
            theta = np.arctan2(y, x)

            c, s = np.cos(theta), np.sin(theta)
            c2, s2, sc = c**2, s**2, s*c

            s_xx, s_yy, s_xy, s_zz = s_tensor[:,0,0], s_tensor[:,1,1], s_tensor[:,0,1], s_tensor[:,2,2]
            sigma_rr = s_xx * c2 + s_yy * s2 + 2 * s_xy * sc
            sigma_tt = s_xx * s2 + s_yy * c2 - 2 * s_xy * sc
            sigma_zz = s_zz

            df = pd.DataFrame({"r": np.round(r, 6), "σ_rr": sigma_rr, "σ_tt": sigma_tt, "σ_zz": sigma_zz})
            grouped = df.groupby("r").mean().reset_index()
            r_avg = grouped["r"].values

            # --- Analytical ---
            import yaml
            with open("geometry.yaml", "r") as f:
                geometry = yaml.safe_load(f)

            if analytical_case == "lame":
                sigma_rr_an, sigma_tt_an, sigma_zz_an, _ = lame_solutions(r_avg)
            elif analytical_case == "mariotte":
                sigma_rr_an, sigma_tt_an, sigma_zz_an = mariotte_solutions(r_avg, geometry, pi=1e6)
                print(f"Radial stress = {np.mean(sigma_rr_an):.3e} Pa")
                print(f"Hoop stress = {np.mean(sigma_tt_an):.3e} Pa")
                print(f"Axial stress = {np.mean(sigma_zz_an):.3e} Pa")
            elif analytical_case == "thermal":
                sigma_rr_an, sigma_tt_an, sigma_zz_an = thermal_stresses(r_avg)
            else:
                sigma_rr_an = sigma_tt_an = sigma_zz_an = np.zeros_like(r_avg)

            # --- Plot ---
            fig, ax = plt.subplots(figsize=(10, 7))
            ax.plot(r_avg, grouped["σ_rr"], 's', c='purple', label=f"σ_rr (FEM, {stress_field_source})")
            ax.plot(r_avg, grouped["σ_tt"], 'v', c='orange', label=f"σ_tt (FEM, {stress_field_source})")
            ax.plot(r_avg, grouped["σ_zz"], '^', c='blue', label=f"σ_zz (FEM, {stress_field_source})")
            ax.plot(r_avg, sigma_rr_an, '--', c='purple', lw=2, label="σ_rr (Analytical)")
            ax.plot(r_avg, sigma_tt_an, '--', c='orange', lw=2, label="σ_tt (Analytical)")
            ax.plot(r_avg, sigma_zz_an, '--', c='blue', lw=2, label="σ_zz (Analytical)")

            ax.set_xlabel("Radius r (m)", fontsize=12)
            ax.set_ylabel("Stress σ (Pa)", fontsize=12)
            ax.set_title(f"'{stress_field_name}' vs. radius at z = {z_level}", fontsize=14)
            ax.grid(True, linestyle=':')
            ax.legend()
            plt.tight_layout()
            plt.show()

        # --- Z-level dropdown for selected field ---
        interact(
            plot_stress_at_z,
            z_level=widgets.Dropdown(
                options=unique_z_coords,
                value=unique_z_coords[len(unique_z_coords) // 2],
                description='Select Z-level:',
                style={'description_width': 'initial'},
                layout=Layout(width='50%')
            )
        )

    # --- Field selection dropdown ---
    interact(
        on_field_selected,
        stress_field_name=widgets.Dropdown(
            options=list(all_stress_fields.keys()),
            description='Select stress field:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

# Call the interactive plot
interactive_stress_plot(
    filename="output/fields.vtu",
    analytical_case="mariotte"
)


In [ ]:
def interactive_temperature_plot(filename="output/fields.vtu",
                                 tol=1e-5,
                                 analytical_case=None):
    """
    Interactive radial temperature plot with dropdown menu for Z-level selection.

    Parameters:
        filename (str): Path to the .vtu file.
        tol (float): Tolerance for selecting Z-level nodes.
        analytical_case (str or None): If provided, plots analytical solution (e.g., 'slab').
    """
    print(f"Loading data from: {filename}")
    grid = pv.read(filename)

    if "Temperature" not in grid.point_data:
        print("[ERROR] Temperature field not found in point_data.")
        return

    points = grid.points
    T_vals = grid.point_data["Temperature"]

    z_coords = np.round(points[:, 2], 6)
    unique_z_coords = np.unique(z_coords)
    print(f"Found {len(unique_z_coords)} unique Z levels.")

    print("\nAvailable Z levels for slicing:")
    for z in unique_z_coords:
        print(f"  → z = {z}")

    def plot_temperature_at_z(z_level):
        mask = np.abs(points[:, 2] - z_level) < tol
        if not np.any(mask):
            print(f"[WARNING] No nodes found at z = {z_level} ± {tol}")
            return

        x, y = points[mask, 0], points[mask, 1]
        r = np.sqrt(x**2 + y**2)
        T = T_vals[mask]

        df = pd.DataFrame({"r": np.round(r, 6), "T": T})
        grouped = df.groupby("r").mean().reset_index()
        r_avg = grouped["r"].values

        # Soluzione analitica opzionale
        if analytical_case == "slab":
            T_analytical = thermal_shield_slab_analytical_temperature(r_avg)
        elif analytical_case == "cylinder":
            # Supponiamo tu abbia una funzione: thermal_cylinder_analytical_temperature(r)
            T_analytical = thermal_cylinder_analytical_temperature(r_avg)
        else:
            T_analytical = None

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(grouped["r"], grouped["T"], 'o-', color='red', label="Temperature (FEM)")

        if T_analytical is not None:
            ax.plot(r_avg, T_analytical, '--k', label="Analytical")

        ax.set_xlabel("Radius r (m)", fontsize=12)
        ax.set_ylabel("Temperature T (K)", fontsize=12)
        ax.set_title(f"Temperature vs. radius at z = {z_level}", fontsize=14)
        ax.grid(True, linestyle=':')
        ax.legend()
        plt.tight_layout()
        plt.show()

    interact(
        plot_temperature_at_z,
        z_level=widgets.Dropdown(
            options=unique_z_coords,
            value=unique_z_coords[len(unique_z_coords) // 2],
            description='Select Z-level:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

interactive_temperature_plot(
    filename="output/fields.vtu",
    analytical_case="lame",
    tol=1e-5
)

In [ ]:
def interactive_displacement_plot(filename="output/fields.vtu",
                                  tol=1e-5,
                                  analytical_case="lame",
                                  mode="cyl"):
    """
    interactive displacement plot with dropdown menu for z-level selection.

    parameters:
        filename (str): path to the .vtu file
        tol (float): tolerance for selecting z-level nodes
        analytical_case (str): analytical case to compare against
        mode (str): 'cyl' for ur, utheta, uz — 'rect' for ux, uy, uz
    """
    print(f"loading data from: {filename}")
    grid = pv.read(filename)

    if "Displacement" not in grid.point_data:
        print("[error] displacement field not found in point_data.")
        return

    points = grid.points
    u = grid.point_data["Displacement"]

    z_coords = np.round(points[:, 2], 6)
    unique_z_coords = np.unique(z_coords)
    print(f"Found {len(unique_z_coords)} unique Z levels.")

    print("\navailable z levels for slicing:")
    for z in unique_z_coords:
        print(f"  → z = {z}")

    def plot_displacement_at_z(z_level):
        mask = np.abs(points[:, 2] - z_level) < tol
        if not np.any(mask):
            print(f"[warning] no nodes found at z = {z_level} ± {tol}")
            return

        x, y = points[mask, 0], points[mask, 1]
        ux, uy, uz = u[mask, 0], u[mask, 1], u[mask, 2]
        r = np.sqrt(x**2 + y**2)
        theta = np.arctan2(y, x)

        if mode == "cyl":
            u_r = ux * np.cos(theta) + uy * np.sin(theta)
            u_t = -ux * np.sin(theta) + uy * np.cos(theta)
            u_z = uz

            df = pd.DataFrame({
                "r": np.round(r, 6),
                "u_r": u_r,
                "u_theta": u_t,
                "u_z": u_z
            })

            grouped = df.groupby("r").mean().reset_index()
            r_avg = grouped["r"].values

            if analytical_case == "lame":
                u_r_an = radial_displacement(r_avg)
            else:
                u_r_an = None

            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(grouped["r"], grouped["u_r"], 'o', color='green', label="u_r (fem)")
            ax.plot(grouped["r"], grouped["u_theta"], 's', color='orange', label="u_theta (fem)")
            ax.plot(grouped["r"], grouped["u_z"], '^', color='blue', label="u_z (fem)")

            # if u_r_an is not None:
            #     ax.plot(r_avg, u_r_an, '--k', label="u_r (analytical)")

            ax.set_ylabel("displacement (m)")
            ax.set_title(f"cylindrical displacement components at z = {z_level}")

        elif mode == "rect":
            df = pd.DataFrame({
                "r": np.round(r, 6),
                "u_x": ux,
                "u_y": uy,
                "u_z": uz
            })

            grouped = df.groupby("r").mean().reset_index()

            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(grouped["r"], grouped["u_x"], 'o', color='green', label="u_x (fem)")
            ax.plot(grouped["r"], grouped["u_y"], 's', color='orange', label="u_y (fem)")
            ax.plot(grouped["r"], grouped["u_z"], '^', color='blue', label="u_z (fem)")

            ax.set_ylabel("displacement (m)")
            ax.set_title(f"cartesian displacement components at z = {z_level}")

        else:
            print(f"[error] mode '{mode}' not recognized. use 'cyl' or 'rect'.")
            return

        ax.set_xlabel("radius r (m)")
        ax.grid(True, linestyle=':')
        ax.legend()
        plt.tight_layout()
        plt.show()

    interact(
        plot_displacement_at_z,
        z_level=widgets.Dropdown(
            options=unique_z_coords,
            value=unique_z_coords[len(unique_z_coords) // 2],
            description='select z-level:',
            style={'description_width': 'initial'},
            layout=Layout(width='50%')
        )
    )

interactive_displacement_plot(filename="output/fields.vtu", mode="cyl")